# Progetto: Tris con Algoritmo Minimax in Python
L'obiettivo è creare un gioco del Tris che permetta sfide tra Umano vs Computer e Computer vs Computer. Il "cervello" della macchina è basato sull'algoritmo Minimax, un modello di teoria dei giochi che permette al computer di non perdere mai.

##  Struttura del gioco
Per rendere il codice ordinato e facile da gestire, abbiamo usato la Programmazione Orientata agli Oggetti (OOP) creando una classe chiamata Tris.

## La Griglia
La scacchiera è rappresentata da una lista di 9 elementi. Gli indici vanno da 0 a 8, partendo dall'alto a sinistra fino in basso a destra.

```Plaintext
 0 | 1 | 2 
-----------
 3 | 4 | 5 
-----------
 6 | 7 | 8 
```

## Logica di Vittoria
Abbiamo implementato un controllo intelligente: ogni volta che viene fatta una mossa, il programma controlla solo la riga, la colonna e le diagonali che passano per quel punto specifico, risparmiando calcoli inutili.

## L'Algoritmo Minimax: La Teoria
Il Minimax è un algoritmo ricorsivo che esplora l'intero "albero delle possibilità" del gioco.

- **Massimizzazione** (Computer): Quando tocca alla macchina, questa cerca la mossa che porta al punteggio più alto (+1).

- **Minimizzazione** (Avversario): La macchina presuppone che l'umano sia perfetto e scelga sempre la mossa che la danneggia di più (-1).

- **Pareggio**: Se la partita finisce senza vincitori, il valore è 0.

L'algoritmo simula ogni singola partita possibile fino alla fine, poi torna indietro (backtracking) scegliendo il percorso ottimale.

In [3]:
#import os
#from IPython.display import clear_output
import time
import random

class Tris:
    def __init__(self):
        # Inizializza la scacchiera vuota e il giocatore iniziale
        self.board = [" " for _ in range(9)] # Rappresenta la griglia 3x3 come una lista di 9 elementi
        self.current_winner = None 

    def stampa_griglia(self):
        #os.system('cls' if os.name == 'nt' else 'clear')
        #clear_output(wait=True)
        # Stampa la griglia in un formato leggibile
        for i in range(0,9,3):
            print(f"{self.board[i]} | {self.board[i+1]} | {self.board[i+2]}")
            if i < 6:
                print("--+---+--")
        print("\n")

    def mosse_disponibili(self):
        # Restituisce una lista degli indici vuoti
        return [i for i, spot in enumerate(self.board) if spot == " "]

    def esegui_mossa(self, quadrante, segno):
        # Assegna il segno (X o O) alla posizione se valida
        if self.board[quadrante] == " ":
            self.board[quadrante] = segno
            # Controlla se questa mossa ha portato alla vittoria
            if self.controlla_vittoria(quadrante, segno):
                self.current_winner = segno
            return True
        return False

    def controlla_vittoria(self, quadrante, segno):
        # Controlla righe, colonne e diagonali rispetto all'ultima mossa
        # Righe
        id_rig = quadrante // 3 # Indice della riga
        riga = self.board[id_rig * 3 : (id_rig + 1) * 3] 
        if all([spot == segno for spot in riga]): 
            return True

        # Colonne
        id_col = quadrante % 3 # Indice della colonna
        col = [self.board[id_col + i * 3] for i in range(3)]
        if all([spot == segno for spot in col]):
            return True
        
        # Diagonali
        if quadrante % 2 == 0:
            diag1 = [self.board[i] for i in [0,4,8]]
            if all([spot == segno for spot in diag1]):
                return True
            
            diag2 = [self.board[i] for i in [2,4,6]]
            if all([spot == segno for spot in diag2]):
                return True
        return False



    def minimax(self, giocatore):
        if self.current_winner == "X": return 1 # Se X ha vinto, restituisce un punteggio positivo
        if self.current_winner == "O": return -1 # Se O ha vinto, restituisce un punteggio negativo
        if not self.mosse_disponibili(): return 0 # Pareggio

        if giocatore == "X": 
            best = -float("inf") # Inizializza il punteggio migliore per X
            for mossa in self.mosse_disponibili(): # Per ogni mossa disponibile
                self.esegui_mossa(mossa, "X") # Esegue la mossa
                punteggio = self.minimax("O") # Chiama minimax per il giocatore avversario (O)
                self.board[mossa] = " " # Backtracking: annulla la mossa
                self.current_winner = None # Resetta il vincitore
                best = max(best, punteggio) # Aggiorna il punteggio migliore per X
            return best # Restituisce il punteggio migliore per X
        else:
            best = float("inf") # Inizializza il punteggio migliore per O
            for mossa in self.mosse_disponibili(): # Per ogni mossa disponibile
                self.esegui_mossa(mossa, "O") # Esegue la mossa
                punteggio = self.minimax("X") # Chiama minimax per il giocatore avversario (X)
                self.board[mossa] = " " # Backtracking: annulla la mossa
                self.current_winner = None # Resetta il vincitore
                best = min(best, punteggio) # Aggiorna il punteggio migliore per O
            return best # Restituisce il punteggio migliore per O
    
    def mossa_pc(self, segno):
        # Restituisce la mossa migliore per l'IA usando minimax
        best_score = -float("inf") if segno == "X" else float("inf") # Inizializza il punteggio migliore in base al segno del PC 
        mosse_migliori = [] # Lista per tenere traccia delle mosse migliori
        avversario = "O" if segno == "X" else "X" # Determina il segno dell'avversario

        for mossa in self.mosse_disponibili(): # Per ogni mossa disponibile
            self.esegui_mossa(mossa, segno) # Esegue la mossa
            punteggio = self.minimax(avversario) # Chiama minimax per valutare la mossa
            self.board[mossa] = " " # Backtracking: annulla la mossa
            self.current_winner = None # Resetta il vincitore

            # Logica per X (Massimizzatore)
            if segno == "X":
                if punteggio > best_score:
                    best_score = punteggio
                    mosse_migliori = [mossa] # Nuovo record, resetto la lista
                elif punteggio == best_score:
                    mosse_migliori.append(mossa) # Mossa equivalente, la aggiungo
        
            # Logica per O (Minimizzatore)
            else:
                if punteggio < best_score:
                    best_score = punteggio
                    mosse_migliori = [mossa]
                elif punteggio == best_score:
                    mosse_migliori.append(mossa)
        return random.choice(mosse_migliori)

In [6]:
def main():
    game = Tris()
    print("Seleziona modalità:\n1. Umano (X) vs Macchina (O)\n2. Macchina (X) vs Macchina (O)")
    scelta = input("Scelta: ")
    
    turno = "X" # X inizia sempre per primo
    while game.mosse_disponibili() and not game.current_winner: # Finché ci sono mosse disponibili e non c'è un vincitore
        game.stampa_griglia() # Stampa la griglia ad ogni turno
        
        if scelta == "1" and turno == "X": # Se è il turno dell'umano (X) e la modalità è 1, chiedi all'utente di inserire una mossa
            try:
                mossa = int(input(f"Turno di {turno}. Scegli (0-8): ")) # Chiede all'utente di inserire un numero da 0 a 8 per la mossa
            except: continue
        else:
            print(f"La macchina ({turno}) sta pensando...")
            mossa = game.mossa_pc(turno) # Ottiene la mossa migliore per il PC
            if scelta == "2": time.sleep(0.8) # Rallenta per vedere le mosse

        if game.esegui_mossa(mossa, turno): # Se la mossa è valida, eseguila e passa al turno successivo
            turno = "O" if turno == "X" else "X" # Cambia il turno

    game.stampa_griglia() # Stampa la griglia finale
    if game.current_winner: # Se c'è un vincitore, stampa chi ha vinto
        print(f"VITTORIA: {game.current_winner}") # Stampa il vincitore
    else:
        print("PAREGGIO!") # Se non c'è un vincitore e non ci sono mosse disponibili, è un pareggio

main()

Seleziona modalità:
1. Umano (X) vs Macchina (O)
2. Macchina (X) vs Macchina (O)
  |   |  
--+---+--
  |   |  
--+---+--
  |   |  


La macchina (X) sta pensando...
  |   |  
--+---+--
  |   | X
--+---+--
  |   |  


La macchina (O) sta pensando...
  |   | O
--+---+--
  |   | X
--+---+--
  |   |  


La macchina (X) sta pensando...
  |   | O
--+---+--
  |   | X
--+---+--
X |   |  


La macchina (O) sta pensando...
  |   | O
--+---+--
  | O | X
--+---+--
X |   |  


La macchina (X) sta pensando...
  |   | O
--+---+--
  | O | X
--+---+--
X | X |  


La macchina (O) sta pensando...
  |   | O
--+---+--
  | O | X
--+---+--
X | X | O


La macchina (X) sta pensando...
X |   | O
--+---+--
  | O | X
--+---+--
X | X | O


La macchina (O) sta pensando...
X |   | O
--+---+--
O | O | X
--+---+--
X | X | O


La macchina (X) sta pensando...
X | X | O
--+---+--
O | O | X
--+---+--
X | X | O


PAREGGIO!
